# SDK prunding


### Local SDK Path


In [7]:
SDK_PATH = '/root/autodl-tmp/revitdocs/Samples'

## Get ReadMe Doc

In [6]:
import os
import json
from striprtf.striprtf import rtf_to_text
from dotenv import load_dotenv
from openai import OpenAI
import google.generativeai as genai

def read_readme_doc(path: str) -> dict:
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # make rtf doc -> text doc 
    plain_text_content = rtf_to_text(content)

    # The f-string prompt with escaped curly braces in the JSON example
    prompt_sdk_select = f"""
        # ROLE
        You are an AI assistant specializing in codebase analysis, an expert at extracting structured data from technical documentation.

        # GOAL
        Your goal is to accurately parse the provided ReadMe file to extract key identifiers for code. This output will be used programmatically by an automated code retrieval and analysis system, so the accuracy and format of your response are critical.

        # INSTRUCTIONS
        1.  Carefully analyze the text provided within the `<ReadMeContent>` tags.
        2.  Extract the following three categories of information:
            - `target_files`: A list of all project source filenames (e.g., `.cs` files) explicitly mentioned in the text.
            - `key_classes_and_methods`: A list of the names of custom classes or methods created *within* the project that are identified as being responsible for core functionality.
            - `mentioned_apis`: A list of key API classes from external frameworks or libraries (e.g., `Autodesk.Revit.DB.View`) that are explicitly listed in the text.
        3.  Format your output as a single, strict JSON object.
        4.  If no information is found for a specific field, its value must be an empty list (`[]`). Do not omit the key from the JSON object.
        5.  Your final response **MUST** contain *only* the raw JSON object, without any explanatory text, markdown code blocks, or other conversational filler.

        # EXAMPLE
        <ExampleReadMe>
        Summary: This tool is in the file `Processor.cs`. The core logic is handled by the `DataParser` class, which uses the `Autodesk.Revit.DB.Transaction` API.
        </ExampleReadMe>
        <ExampleJSONOutput>
        {{
        "target_files": ["Processor.cs"],
        "key_classes_and_methods": ["DataParser"],
        "mentioned_apis": ["Autodesk.Revit.DB.Transaction"]
        }}
        </ExampleJSONOutput>

     
    """

    # initialize openai
    load_dotenv(dotenv_path='/root/autodl-tmp/python_revit_train/gemini_api.env')
    # Corrected environment variable name for consistency
    deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
    if not deepseek_api_key:
        raise ValueError("Error: DEEPSEEK_API_KEY environment variable not set.")
        
    #genai.configure(api_key=gemini_api_key)

    #model = genai.GenerativeModel('gemini-1.5-flash-latest')

    #response = model.generate_content(prompt_sdk_select)

    # Create Response
    client = OpenAI(api_key=deepseek_api_key , base_url="https://api.deepseek.com")

    response = client.chat.completions.create(
    model="deepseek-chat",
        messages=[
            {"role": "system", "content": prompt_sdk_select},  
            
            {"role": "user", "content": f"{plain_text_content}"}
        ],
        stream = False
    )

    # response = gemini_model.generate_content(query_llm)

    # print(f"query: {query_llm}")
    print("Response from DeepSeek:")
    print(response.choices[0].message.content)



    cleaned_json_string = response.choices[0].message.content.strip().replace("```json", "").replace("```", "").strip()

    return json.loads(cleaned_json_string)


if __name__ == "__main__":
    # Ensure the striprtf library is installed: pip install striprtf
    target_content = read_readme_doc('/root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf')
    print(json.dumps(target_content, indent=2, ensure_ascii=False))

Response from DeepSeek:
{
  "target_files": ["AllViews.cs", "AllViewsForm.cs"],
  "key_classes_and_methods": ["Command", "ViewsMgr", "AllViewsForm"],
  "mentioned_apis": ["Autodesk.Revit.DB.View", "Autodesk.Revit.DB.ViewSet", "Autodesk.Revit.Creation.Document.NewViewSheet"]
}
{
  "target_files": [
    "AllViews.cs",
    "AllViewsForm.cs"
  ],
  "key_classes_and_methods": [
    "Command",
    "ViewsMgr",
    "AllViewsForm"
  ],
  "mentioned_apis": [
    "Autodesk.Revit.DB.View",
    "Autodesk.Revit.DB.ViewSet",
    "Autodesk.Revit.Creation.Document.NewViewSheet"
  ]
}


## Get This Project Most Important Code Setences

#### revit sdk sampl code

In [8]:
csharp_code_revit = """
//
// (C) Copyright 2003-2023 by Autodesk, Inc. All rights reserved.
//
// Permission to use, copy, modify, and distribute this software in
// object code form for any purpose and without fee is hereby granted
// provided that the above copyright notice appears in all copies and
// that both that copyright notice and the limited warranty and
// restricted rights notice below appear in all supporting
// documentation.

//
// AUTODESK PROVIDES THIS PROGRAM 'AS IS' AND WITH ALL ITS FAULTS.
// AUTODESK SPECIFICALLY DISCLAIMS ANY IMPLIED WARRANTY OF
// MERCHANTABILITY OR FITNESS FOR A PARTICULAR USE. AUTODESK, INC.
// DOES NOT WARRANT THAT THE OPERATION OF THE PROGRAM WILL BE
// UNINTERRUPTED OR ERROR FREE.
//
// Use, duplication, or disclosure by the U.S. Government is subject to
// restrictions set forth in FAR 52.227-19 (Commercial Computer
// Software - Restricted Rights) and DFAR 252.227-7013(c)(1)(ii)
// (Rights in Technical Data and Computer Software), as applicable. 

using System;
using System.Windows.Forms;

using Autodesk.Revit.UI;

using TaskDialog = Autodesk.Revit.UI.TaskDialog;

namespace APIAppStartup
{
   [Autodesk.Revit.Attributes.Transaction(Autodesk.Revit.Attributes.TransactionMode.Manual)]
   [Autodesk.Revit.Attributes.Regeneration(Autodesk.Revit.Attributes.RegenerationOption.Manual)]
   [Autodesk.Revit.Attributes.Journaling(Autodesk.Revit.Attributes.JournalingMode.NoCommandData)]
   public class AppSample : IExternalApplication
   {
      #region IExternalApplication Members

      public Autodesk.Revit.UI.Result OnShutdown(UIControlledApplication application)
      {
         TaskDialog.Show("Revit", "Quit External Application!");
         return Autodesk.Revit.UI.Result.Succeeded;
      }

       public Autodesk.Revit.UI.Result OnStartup(UIControlledApplication application)
      {
         String version = application.ControlledApplication.VersionName;

         //display splash window for 10 seconds
         SplashWindow.StartSplash();
         SplashWindow.ShowVersion(version);
         System.Threading.Thread.Sleep(10000);
         SplashWindow.StopSplash();

         return Autodesk.Revit.UI.Result.Succeeded;
      }

      #endregion
   }
}


"""

### tree-sitter


In [4]:
from tree_sitter import Language , Parser , Query ,Node
import tree_sitter_c_sharp
import collections

def ini_query(content : str) :
    CSHARP_LANGUAGE =  Language(tree_sitter_c_sharp.language())
    # this is a easy code to get value
    csharp_code_for_query = """
    public class Calculator
    {
        public int Add(int x, int y) => x + y;
        private static string GetWelcomeMessage() => "Welcome!";
    }
    """

    cpp_parser = Parser(CSHARP_LANGUAGE)


    tree = cpp_parser.parse(bytes(content, "utf8"))
    root_node = tree.root_node
    return CSHARP_LANGUAGE , root_node

def get_details_query(class_name : str , root_node : Node , lang : Language) -> list:
    """
    input class_name that get all method context 
    
    """
    # 定义一个查询字符串 | get a main query to get all code method in class
    # - 查找所有 method_declaration 节点 | find all method_declaration block
    # - 在该节点下，捕获返回类型 (predefined_type 或 identifier) 并命名为 @return.type in this block , get return type and named to @return.type
    # - 捕获方法名 (identifier) 并命名为 @method.name | in this block get method name and named to @method.name
    # https://tree-sitter.github.io/tree-sitter/7-playground.html this is a online website that to check query structure
    query_string = f"""
        (compilation_unit
            (namespace_declaration
                body: (declaration_list
                    (class_declaration
                        name: (identifier) @class.name
                        (base_list) @base.list.name?
                        (#any-of? @class.name {class_name})
                        body: (declaration_list
                        (method_declaration) @method.node
                        )
                    )
                )
            )
        )
    """
    print(f'first query str : {query_string}')
    # get the method query result 
    query = Query(lang, query_string)

    # 对语法树执行查询
    # captures = query.captures(root_node)
    # print(captures)

    # use matches to get all code and return a tuple[int , dic[int , list[node]]]
    matches = query.matches(root_node) # return a tuple
    final_methods_list = []
    for match in matches:
        # the second query to split the parameters , this can get muti-parameter in method 
        query_sub_string = """
        (
                method_declaration
                returns: (_) @return.type
                name: (identifier) @method.name
                parameters: (parameter_list
                    (parameter) @param.complete
                )*
                body : (_) @method.body
        )
        """
        # get the target node which has method type and name 
        print('start sub query ')
        values = match[1]
        value_node = values['method.node'][0]
        get_details_query = Query(lang, query_sub_string)
        detail_captures = get_details_query.captures(value_node) # return a dictionary

    
        # define a struct : name , return type and params
        method_details = {
                "name": "",
                "return_type": "void",
                "params": []  , # this is a params group
                "body" : ""
            }
        
                
        # group to params
        param_nodes = []
        # details_captures is a dictionary so need use items
        for name , node in detail_captures.items():
            if name == 'method.name':
                method_details['name'] = node[0].text.decode('utf8')
            elif name == 'return.type':
                method_details['return_type'] = node[0].text.decode('utf8')
            elif name == 'param.complete':
                # 将找到的完整参数节点添加到临时列表中
                for sub_node in node :
                    param_nodes.append(sub_node)
            elif name == 'method.body' :
                method_details['method.body'] = node[0].text.decode('utf-8')
                    
            # union the paramas value
        for param_node in param_nodes:
            method_details['params'].append(param_node.text.decode('utf8'))
                
        final_methods_list.append(method_details)
        
    # prinf
    #print(final_methods_list)
    return final_methods_list


if __name__ == "__main__" :
    configs = ini_query(csharp_code_revit)
    l = get_details_query("AppSample SplashWindow OnStartup OnShutdown" , configs[1] , configs[0])
    print(l)


first query str : 
        (compilation_unit
            (namespace_declaration
                body: (declaration_list
                    (class_declaration
                        name: (identifier) @class.name
                        (base_list) @base.list.name?
                        (#any-of? @class.name AppSample SplashWindow OnStartup OnShutdown)
                        body: (declaration_list
                        (method_declaration) @method.node
                        )
                    )
                )
            )
        )
    
start sub query 
start sub query 
[{'name': 'OnShutdown', 'return_type': 'Autodesk.Revit.UI.Result', 'params': ['UIControlledApplication application'], 'body': '', 'method.body': '{\n         TaskDialog.Show("Revit", "Quit External Application!");\n         return Autodesk.Revit.UI.Result.Succeeded;\n      }'}, {'name': 'OnStartup', 'return_type': 'Autodesk.Revit.UI.Result', 'params': ['UIControlledApplication application'], 'body': '', 

In [ ]:
import os
from typing import List , Dict , Any

def get_all_files(root_path : str)  -> List[Dict[str, Any]]:
    """
    扫描一个根目录，找到所有项目文件夹，并为每个项目找到ReadMe.rtf和所有文件的路径。

    Args:
        root_path: 要扫描的根目录路径 (例如: '/root/autodl-tmp/revitdocs/Samples/')。

    Returns:
        一个项目信息列表。每个项目是一个字典，包含:
        - 'project_name': 项目文件夹的名称。
        - 'project_path': 项目文件夹的完整路径。
        - 'readme_path': 'ReadMe.rtf' 文件的完整路径 (如果找到的话，否则为 None)。
        - 'all_files': 项目中所有文件的完整路径列表。
    """
   
    if not os.path.isdir(root_path):
            print(f"❌ 错误：提供的根路径 '{root_path}' 不是一个有效的目录。")
            return []

    projects_list = []
    print(f"🚀 开始高级扫描根目录: {root_path}")

    # os.walk() 会自顶向下地遍历整个目录树
    for dirpath, dirnames, filenames in os.walk(root_path):
            
            # --- 新需求 2: 过滤VB.NET项目 ---
            # 检查当前文件夹是否包含任何 .vb 文件
            has_vb = any(f.lower().endswith('.vb') for f in filenames)
            if has_vb:
                print(f"⏭️  跳过VB.NET项目: {dirpath}")
                # 清空dirnames列表，告诉os.walk不要再深入这个目录的任何子目录
                dirnames[:] = [] 
                continue

            # --- 新需求 1: 识别C#项目 ---
            # 检查当前文件夹是否是 'CS' 文件夹，或者直接包含 .cs 文件
            is_cs_folder = os.path.basename(dirpath).lower() == 'cs'
            has_cs_files = any(f.lower().endswith('.cs') for f in filenames)

            if is_cs_folder or has_cs_files:
                print(f"🎯 发现C#源码目录: {dirpath}")

                # --- 新需求 1: 确定项目逻辑根目录和项目名称 ---
                if is_cs_folder:
                    # 如果是'CS'文件夹，则其父目录是项目的逻辑根目录
                    project_root = os.path.dirname(dirpath)
                else:
                    # 否则，当前目录就是项目的逻辑根目录
                    project_root = dirpath
                
                # 计算相对于扫描根目录的路径，并生成项目名称
                relative_path = os.path.relpath(project_root, root_path)
                project_name = relative_path.replace(os.sep, '.')
                
                print(f"   -> 项目名称: {project_name}")
                print(f"   -> 项目根目录: {project_root}")

                # --- 搜集项目信息 ---
                project_data = {
                    "project_name": project_name,
                    "project_path": project_root,
                    "readme_path": None,
                    "all_files": []
                }

                # 再次遍历项目根目录，以收集所有文件和ReadMe
                for proj_dirpath, _, proj_filenames in os.walk(project_root):
                    for proj_filename in proj_filenames:
                        full_path = os.path.join(proj_dirpath, proj_filename)
                        project_data["all_files"].append(full_path)
                        
                        if 'readme' in  proj_filename.lower():
                            project_data["readme_path"] = full_path

                if project_data["readme_path"]:
                    print(f"   -> ✅ 找到 ReadMe 文件: {project_data['readme_path']}")
                else:
                    print(f"   -> ⚠️ 警告: 在项目 '{project_name}' 中未找到 'ReadMe.rtf'。")

                projects_list.append(project_data)
                
                # 告诉os.walk不要再深入这个已识别项目的任何子目录，避免重复
                dirnames[:] = []
                
    print("\n扫描完成！")
    return projects_list



def find_file_path_in_project(project_data: Dict[str, Any], target_filename: str) -> str | None:
    """
    在一个项目的数据字典中，根据文件名查找其完整的路径。

    Args:
        project_data: 包含项目信息的字典，必须含有 'all_files' 键。
        target_filename: 您要查找的文件的名字 (例如: "AllViews.cs")。

    Returns:
        如果找到文件，则返回其完整的路径字符串；如果未找到，则返回 None。
    """
    # 检查 'all_files' 键是否存在并且是一个列表
    if 'all_files' not in project_data or not isinstance(project_data['all_files'], list):
        print("错误：'project_data' 字典中没有找到 'all_files' 列表。")
        return None

    # 遍历项目中的每一个文件的完整路径
    for full_path in project_data['all_files']:
        # 从完整路径中提取文件名
        # os.path.basename() 可以正确处理 'A/B/C.txt' -> 'C.txt'
        if os.path.basename(full_path) == target_filename:
            # 如果文件名匹配，则返回这个完整路径
            return full_path
    
    # 如果遍历完所有文件都没有找到，则返回 None
    return None




# --- 使用示例 ---
if __name__ == "__main__":
    # 请将此路径替换为您的Revit SDK Samples的根目录
    SDK_ROOT = '/root/autodl-tmp/revitdocs/Samples' 
    
    # 执行高级扫描
    found_projects = get_all_files(SDK_ROOT)
    
    if found_projects:
       for project in found_projects :
            target_content = read_readme_doc(project["readme_path"])
            if target_content :
                 print(target_content)
                 target_files = target_content.get('target_files')
                 target_class_method = target_content.get('key_classes_and_methods')
                 if target_class_method :
                    print('target class string is ========>')
                    target_class_method_str = " ".join(target_class_method)
                    print(target_class_method_str)
                 for target_file in target_files :
                    print(f'open file : {target_file}')
                    find_file = find_file_path_in_project(project , target_file)
                    print(find_file)
                    with open(find_file, 'r', encoding='utf-8-sig', errors='ignore') as f:
                        file_content = f.read()
                    print('Ini with code content')
                    configs = ini_query(content=file_content)
                    res = get_details_query(target_class_method_str , configs[1] , configs[0])
                    if res :
                        print(f'result is : {res}')
                 break


🚀 开始高级扫描根目录: /root/autodl-tmp/revitdocs/Samples
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS
   -> 项目名称: APIAppStartup
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/APIAppStartup
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/APIAppStartup/CS/ReadMe_APIAppStartup.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AllViews/CS
   -> 项目名称: AllViews
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AllViews
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS
   -> 项目名称: AnalysisVisualizationFramework.DistanceToSurfaces
   -> 项目根目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces
   -> ✅ 找到 ReadMe 文件: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramework/DistanceToSurfaces/CS/ReadMe_DistanceToSurfaces.rtf
🎯 发现C#源码目录: /root/autodl-tmp/revitdocs/Samples/AnalysisVisualizationFramewo

### The Full Agent Workflow

In [12]:
import json
import os
from tree_sitter import Language , Parser

def full_agent_flow( root_path : str) :
    configs = ini_query()

    readme_path = os.path.join(root_path , 'ReadMe_AllViews.rtf')
    print('=====================================')
    print('Agent Start')
    target_content = read_readme_doc(readme_path)
    if not target_content :
        print("Fail To Use This Tools : read_readme_doc")
        return None
    print(json.dumps(target_content, indent=2, ensure_ascii=False))
    all_class_details = []

    for class_name in target_content.get('key_classes_and_methods' , []) :
        class_found = False
        for file_name in target_content.get('target_files' , []):
            full_path = os.path.join(root_path , file_name)

            class_detail = get_details_query(class_name , configs[1] , configs[0])

            if class_detail :
                all_class_details.append(class_detail)
                class_found = True
                break

        if not class_found :
            print(f'cant find this class_name : {class_name}')

    print('\n Agent ===> Compare Data')

    comparehensive_data = {
        "project_name" : readme_path ,
        "readme_summary" : target_content ,
        "detail_code_analysis" : all_class_details
    } 

    print("Agent Done ")
    return comparehensive_data


if __name__ == "__main__" :
    sdk_root_path = '/root/autodl-tmp/revitdocs/Samples/AllViews/CS'

    final_report = full_agent_flow(sdk_root_path)

    if final_report :
        print(json.dumps(final_report , indent=2 , ensure_ascii=False))


Agent Start
Response from DeepSeek:
{
  "target_files": ["AllViews.cs", "AllViewsForm.cs"],
  "key_classes_and_methods": ["Command", "ViewsMgr", "AllViewsForm"],
  "mentioned_apis": ["Autodesk.Revit.DB.View", "Autodesk.Revit.DB.ViewSet", "Autodesk.Revit.Creation.Document.NewViewSheet"]
}
{
  "target_files": [
    "AllViews.cs",
    "AllViewsForm.cs"
  ],
  "key_classes_and_methods": [
    "Command",
    "ViewsMgr",
    "AllViewsForm"
  ],
  "mentioned_apis": [
    "Autodesk.Revit.DB.View",
    "Autodesk.Revit.DB.ViewSet",
    "Autodesk.Revit.Creation.Document.NewViewSheet"
  ]
}
cant find this class_name : AllViewsForm

 Agent ===> Compare Data
Agent Done 
{
  "project_name": "/root/autodl-tmp/revitdocs/Samples/AllViews/CS/ReadMe_AllViews.rtf",
  "readme_summary": {
    "target_files": [
      "AllViews.cs",
      "AllViewsForm.cs"
    ],
    "key_classes_and_methods": [
      "Command",
      "ViewsMgr",
      "AllViewsForm"
    ],
    "mentioned_apis": [
      "Autodesk.Revit.DB.View